# Speech Synthesis (TTS): Acoustic Models, FastSpeech Length Regulation & Vocoding

This interactive lab covers:
1. **Modern Two-Stage TTS Paradigm**: Text $\to$ Intermediate Representation (Mel Spectrogram) $\to$ Waveform.
2. **FastSpeech Length Regulation**: Non-autoregressive expansion of phoneme states to acoustic frame rates.
3. **Text-to-Mel Acoustic Modeling**: Feedforward encoder-decoder predicting log-Mel energy bins.
4. **Vocoding & Phase Retrieval**: Inverting magnitude spectrograms to audible waveforms via the Griffin-Lim algorithm.
5. **Spectral Synthesis Evaluation**: Mel-Cepstral Distortion (MCD) in decibels (dB).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('05-speech-audio/speech-synthesis-tts/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import torch
from tts_engine import (
    LengthRegulator,
    MiniAcousticTTS,
    griffin_lim_reconstruction,
    compute_mcd,
)

print('Speech Synthesis (TTS) engine loaded successfully!')

## 1. FastSpeech Length Regulation Mechanism

Expanding discrete phoneme tokens by variable duration counts to match acoustic frame rate.

In [ ]:
regulator = LengthRegulator()

# 3 phoneme vectors of dimension 8
phoneme_reps = torch.tensor([[ 
    [1.0]*8,  # Phoneme /k/
    [2.0]*8,  # Phoneme /ae/
    [3.0]*8,  # Phoneme /t/
]])
# Durations: /k/ lasts 2 frames, /ae/ lasts 4 frames, /t/ lasts 3 frames
durations = torch.tensor([[2, 4, 3]])

expanded = regulator(phoneme_reps, durations)
print(f'Phoneme sequence length: {phoneme_reps.size(1)}')
print(f'Expanded acoustic frames: {expanded.size(1)} (sum: {durations.sum().item()})')
assert expanded.shape == (1, 9, 8)

## 2. Text-to-Mel Acoustic Model Forward Pass

Generating a 40-channel Mel spectrogram from input phoneme IDs.

In [ ]:
torch.manual_seed(42)
tts_model = MiniAcousticTTS(vocab_size=20, embed_dim=16, num_mels=40)

token_ids = torch.tensor([[2, 8, 14, 5]])  # Text sequence: 4 phonemes
ground_truth_mel = torch.randn(1, 12, 40)  # Target acoustic frames
target_durations = torch.tensor([[3, 4, 2, 3]])  # Total = 12 frames

out = tts_model(token_ids, target_durations=target_durations, target_mel=ground_truth_mel)
print(f'Predicted Mel Spectrogram shape: {out["mel_out"].shape}')
print(f'L1 Acoustic Spectrogram Loss:   {out["loss"].item():.4f}')

## 3. Vocoding: Griffin-Lim Phase Retrieval & Waveform Inversion

Estimating phase angles iteratively to reconstruct a time-domain waveform from a magnitude spectrogram.

In [ ]:
# Synthetic positive magnitude spectrogram (257 FFT bins x 20 frames)
mag_spec = np.abs(np.random.randn(257, 20)).astype(np.float32)
waveform = griffin_lim_reconstruction(mag_spec, n_fft=512, hop_length=160, win_length=400, n_iter=8)

print(f'Reconstructed waveform samples: {len(waveform)}')
print(f'Audio duration at 16 kHz:       {len(waveform) / 16000.0:.3f} seconds')
print(f'Audio amplitude range:          [{waveform.min():.3f}, {waveform.max():.3f}]')

## 4. Evaluation: Mel-Cepstral Distortion (MCD)

Measuring spectral fidelity between synthesized and reference Mel spectrograms.

In [ ]:
mel_pred = out['mel_out'].detach().squeeze(0).transpose(0, 1).numpy()
mel_target = ground_truth_mel.squeeze(0).transpose(0, 1).numpy()

mcd_score = compute_mcd(mel_pred, mel_target)
print(f'Mel-Cepstral Distortion (MCD): {mcd_score:.2f} dB')
print(f'(Lower MCD indicates higher spectral synthesis fidelity)')